# Kaggle runner — DL-PRJ04

Lanza `src.train` / `src.evaluate` en GPU (T4 preferible). No es el producto: solo clona el repo, instala deps, monta datasets y corre el mismo comando que en la laptop.

**Datasets esperados** (Add data):
- `asvpoof-2019-dataset` (público)
- opcionales: `asvspoof2021-la`, `mlaad-es`, `voces-propias`

In [ ]:
# === Parámetros de la corrida ===
REPO_URL = "https://github.com/<ORG_O_USER>/DL-PRJ04.git"  # cambiar
BRANCH = "main"
CONFIG = "configs/lfcc_cnn.yaml"   # A1; A2: lfcc_cnn_aug; B1: aasist_l; B2: aasist_l_aug
RUN_NAME = "A1"
MACHINE = "kaggle"
EPOCHS = None   # None = usa el del config
RESUME = False  # True para retomar desde /kaggle/working/runs/<RUN_NAME>
EVAL_SPLITS = "asv19_dev asv19_eval"  # tras entrenar

In [ ]:
import os, sys, subprocess
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "DL-PRJ04"

if not REPO.exists():
    subprocess.check_call(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO)])
else:
    subprocess.check_call(["git", "-C", str(REPO), "fetch", "--all"])
    subprocess.check_call(["git", "-C", str(REPO), "checkout", BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only"])

os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

In [ ]:
# PyTorch en Kaggle ya trae CUDA; instalamos el resto del pyproject sin forzar cu128.
%pip install -q numpy pandas scikit-learn soundfile librosa pyyaml tqdm tensorboard matplotlib \
    huggingface-hub captum onnx onnxruntime fastapi "uvicorn[standard]" python-multipart transformers

In [ ]:
from pathlib import Path
import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

# Genera protocolos ASVspoof 2019 si no están (o regenera contra la ruta de Kaggle).
ASV19 = Path("/kaggle/input/asvpoof-2019-dataset/LA/LA")
assert ASV19.is_dir(), f"Monta el dataset asvpoof-2019-dataset; no está {ASV19}"
!python scripts/build_protocols.py --root {ASV19}

In [ ]:
import subprocess
cmd = ["python", "-m", "src.train", "--config", CONFIG, "--machine", MACHINE, "--name", RUN_NAME]
if RESUME:
    cmd = ["python", "-m", "src.train", "--resume", f"/kaggle/working/runs/{RUN_NAME}"]
elif EPOCHS:
    cmd += ["--epochs", str(EPOCHS)]
print(" ".join(cmd))
subprocess.check_call(cmd)

In [ ]:
ckpt = f"/kaggle/working/runs/{RUN_NAME}/best.pt"
!python -m src.evaluate --checkpoint {ckpt} --splits {EVAL_SPLITS}
print("Copia results/ y runs/ desde /kaggle/working (y el repo) como output del notebook.")